# Tầng 1: sinh mô tả cho tune CONSTRUCT (Kaggle **GPU T4 ×2**, Omni-3B, MER2025-FG 1.000 clip)
Kế hoạch: `ideas/F1_tuning_plan_kaggle3b_2026-11-28.md`. Chạy **sau tầng 0**. Lịch chạy trong ô dưới sẽ được Claude chỉnh lại theo số đo thời gian của tầng 0.

Mỗi session chạy các job ghi trong `JOBS`. Mỗi GPU chạy lần lượt các job của nó; mỗi job là "cấu hình : shard":
- S0 sinh 32 mẫu + greedy;
- S1, S2, S3 sinh 16 mẫu.

Ngân sách 10 giờ, dừng cứng ở 10,5 giờ. Notebook snapshot `t1_out_<tên>.tar` mỗi 20 phút.

**Chạy tiếp khi hết giờ:** Save Version, rồi gắn output của version đó làm **Input** của chính notebook này và chạy lại với cùng `JOBS`. Notebook tự chép lại các file `desc_*.jsonl` cũ và bỏ qua các clip đã xong.

**Lịch chạy** (theo số đo của tầng 0 lần 2, commit `0ac2d39`, 12 clip; trung bình mỗi clip trên 1 T4):
- S0 (32 mẫu + greedy): 247 giây;
- S1: 116 giây;
- S2: 122 giây.

Mỗi GPU nhận 1/4 số clip của mỗi cấu hình, chạy theo thứ tự **S1 → S0 → S2**. Thứ tự này để có sớm cặp so sánh S0–S1.

Mỗi GPU cần khoảng 8,1 + 17,2 + 8,5 = 33,8 giờ, tức **4 session cho mỗi notebook**.

| Notebook | `NAME` | GPU0 | GPU1 |
|---|---|---|---|
| A | `A` | S1:0/4 → S0:0/4 → S2:0/4 | S1:1/4 → S0:1/4 → S2:1/4 |
| B | `B` | S1:2/4 → S0:2/4 → S2:2/4 | S1:3/4 → S0:3/4 → S2:3/4 |

**Chạy lại cho tới khi xong:**
- Mỗi notebook giữ nguyên `JOBS`.
- Trước mỗi session, gắn output của **version trước của chính nó** làm Input.
- Job nào đã xong thì tự bỏ qua; job dở thì chạy tiếp từ clip chưa làm.
- Gửi em file `t1_out_<NAME>.tar` sau **mỗi** session. Em bắt đầu phân tích S0–S1 ngay khi đủ dữ liệu.

S3 (ε-sampling) tạm hoãn. Lý do: ở tầng 0, S3 có tỉ lệ mẫu rỗng và mẫu chỉ có từ ngoài từ vựng cao nhất (1,6% và 3,6%). Đây là đánh giá trên chỉ 12 clip.

Cài đặt: GPU T4 ×2, Internet On, secret **HF_TOKEN** (thêm **MER2026_ZIP_PW** nếu được hỏi).
**Gửi lại cho Claude:** file `t1_out_<tên>.tar` của mỗi notebook.

In [ ]:
NAME = 'A'                                  # A / B / C ... (names the archive)
JOBS = {0: ['S1:0/4', 'S0:0/4', 'S2:0/4'], 1: ['S1:1/4', 'S0:1/4', 'S2:1/4']}   # notebook A; notebook B: shards 2/4 and 3/4 (table above)
BUDGET_H, HARD_H = 10.0, 10.5

KOUT, WORK = '/kaggle/working', '/tmp/t1'
import os, sys, time, json, glob, subprocess
T0 = time.time()
os.makedirs(WORK, exist_ok=True)
subprocess.run(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme', shell=True, check=True)
COMMIT = subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout.strip()
print('code commit:', COMMIT)
OV = f'{WORK}/icme/explore/ovmer'; sys.path.insert(0, OV)
import kaggle_t1 as K
K.secrets()
ENV = K.setup(WORK)

In [ ]:
ITEMS = K.prepare(WORK, models=(K.OMNI,))
K.restore(KOUT)                              # outputs of earlier sessions attached as inputs
K.sh(f'df -h {WORK} | tail -1')

In [ ]:
left = BUDGET_H - (time.time() - T0) / 3600; hard = HARD_H - (time.time() - T0) / 3600
print(f'run budget {left:.2f} h (hard stop {hard:.2f} h)')
K.run_queues(JOBS, ITEMS, KOUT, budget_h=left, hard_h=hard, t0=time.time(), tar_name=f't1_out_{NAME}.tar')

In [ ]:
DESCS = sorted(glob.glob(f'{KOUT}/desc_S*.jsonl'))
subprocess.run([sys.executable, f'{OV}/t1_inspect.py', '--descs', ','.join(DESCS), '--n_clips', '1000', '--gpus', '2',
                '--report', f'{KOUT}/t1_report_{NAME}.md'], check=True)
for d in DESCS:
    ok = sum(1 for l in open(d) if json.loads(l).get('ok')); print(os.path.basename(d), 'clips ok:', ok)
K.snapshot(KOUT, f't1_out_{NAME}.tar'); print('total', round((time.time() - T0) / 3600, 2), 'h')